**Reference Link:** [RAG Systems Essentials (Analytics Vidhya)](https://courses.analyticsvidhya.com/courses/take/rag-systems-essentials/lessons/60148017-hands-on-deep-dive-into-rag-evaluation-metrics-generator-metrics-i)

# Retriever Evaluation Metrics

## Overview
- This notebook demonstrates how to evaluate RAG system retrievers using DeepEval metrics
- It focuses on three key evaluation metrics for assessing retrieval quality in RAG pipelines

## Key Evaluation Metrics

### **Contextual Precision**
- **Purpose**: Measures whether relevant document chunks are ranked higher than irrelevant ones
- **Input Requirements**: Query, actual output, expected output, and retrieval context
- **Scoring**: Evaluates ranking quality of retrieved documents (0.0 to 1.0)
- **Use Case**: Assesses how well the retriever prioritizes relevant information

### **Contextual Recall**
- **Purpose**: Measures how well the retrieval context aligns with expected output
- **Input Requirements**: Query, actual output, expected output, and retrieval context
- **Scoring**: Evaluates coverage of expected information in retrieved documents
- **Use Case**: Determines if retriever captures all necessary information

### **Contextual Relevancy**
- **Purpose**: Measures overall relevance of retrieved context for a given query
- **Input Requirements**: Query, actual output, and retrieval context
- **Scoring**: Evaluates general relevance of all retrieved information
- **Use Case**: Assesses overall quality of retrieved content

## Technical Implementation

- **DeepEval Framework**: Uses DeepEval's LLM-based evaluation metrics
- **LLM Judge**: GPT-4o model evaluates relevance and provides reasoning
- **Test Cases**: Creates LLMTestCase objects for systematic evaluation
- **Thresholds**: Configurable success thresholds (default: 0.5)
- **Verbose Mode**: Provides detailed reasoning for metric scores

## Evaluation Process

1. **Setup**: Run existing RAG pipeline to get retrieval results
2. **Context Preparation**: Extract and format retrieved documents
3. **Metric Configuration**: Set up evaluation parameters and thresholds
4. **Testing**: Run evaluation on test cases with different contexts
5. **Analysis**: Review scores, reasons, and pass/fail results

## Benefits

- **Quality Assurance**: Systematic evaluation of retrieval performance
- **Debugging**: Identifies issues with document ranking and relevance
- **Optimization**: Provides metrics to improve retriever performance
- **Transparency**: Clear reasoning for evaluation scores

In [1]:
%run Build_RAG_Pipeline_with_Source.ipynb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.11 requires langchain<0.4.0,>=0.3.11, but you have langchain 0.3.10 which is incompatible.


Metadata: {'id': 10, 'title': 'Artificial Intelligence'}
Content Brief:


Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.


Metadata: {'id': 10, 'title': 'Artificial Intelligence'}
Content Brief:


Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.

...

NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.


Metadata: {'id': 5, 'title': 'Photosynthesis'}
Content Brief:


Photosynthesis is the process plants use to convert sunlight into energy. This process produces glucose and releases oxygen as a byproduct. It is crucial for sustaining life on Earth by providing food and oxygen.


Metadata: {'id': 5, 'title': 'Photosynthesis'}
Content Brief:


Photosynthesis is the process plants use to convert sunlight into energy. This process produces glucose and releases oxygen as a byproduct. It is crucial for sustaining life on Earth by providing food and oxygen.

# Retriever Evaluation Metrics

![](https://i.imgur.com/5S4FhMB.png)

The retrieval process generally includes these steps:

- Convert the initial input query into an embedding using an embedding model of your choice (e.g., OpenAI's `text-embedding-3` model).
- Conduct a vector search with the embedded input on a vector database that holds your vectorized knowledge base, retrieving the top-K most "similar" document chunks.
- Optionally user a Reranker to rerank the retrieved results


Key Metrics to Evaluate here include:

- Contextual Precision
- Contextual Recall
- Contextual Relevancy

## Contextual Precision

The contextual precision metric measures your RAG pipeline's retriever by evaluating whether document chunks (nodes) in your `retrieval_context` that are relevant to the given `input` are ranked higher than irrelevant ones.

`deepeval`'s contextual precision metric is a self-explaining LLM-Eval, meaning it outputs a reason for its metric score using an LLM as a judge.

In `deepeval`, to use the ContextualPrecisionMetric, you'll have to provide the following arguments when creating an `LLMTestCase`:

- `input` : Input Query
- `actual_output` : Actual LLM Response (not used in the computation)
- `expected_output` : Expected LLM Response (ground truth answer)
- `retrieval_context` : Top-N retrieved document chunks (nodes) from Vector DB


![](https://i.imgur.com/oVwrRAU.png)





In [2]:
query = "What is AI?"
response = rag_chain_w_sources.invoke(query)
response

{'context': [Document(id='ca8293b0-0909-4d6d-81b3-1b1568851c4e', metadata={'id': 10, 'title': 'Artificial Intelligence'}, page_content="Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning."),
  Document(id='453580ed-61d6-4f43-a237-7432eebc2be5', metadata={'title': 'Artificial Intelligence', 'id': 10}, page_content="Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning."),
  Document(id='69522413-c9ae-41cf-b1b5-cc4e288dc58e', metadata={'id': 3, 'title': 'Natural Language Processing (NLP)'}, page_content='NLP is a branch of AI that enables computers to underst

### Example:

In [3]:
retrieved_context = [doc.page_content for doc in response['context']]
retrieved_context

["Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.']

In [4]:
human_answer = """AI, also known as Artificial Intelligence is used to build complex systems for applications
                  like virtual assistants, robotics and autonomous vehicles."""

In [5]:
new_context = ['Machine Learning is the study of algorithms which learn with more data',
               'AI is known as Artificial Intelligence'] + retrieved_context
new_context

['Machine Learning is the study of algorithms which learn with more data',
 'AI is known as Artificial Intelligence',
 "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.']

In [6]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric
from deepeval import evaluate

test_case = LLMTestCase(
    input=response['question'],
    actual_output=response['response'],
    expected_output=human_answer,
    retrieval_context=new_context
)

metric = ContextualPrecisionMetric(
    threshold=0.5,
    model="gpt-4o",
    include_reason=True,
    verbose_mode=True
)

result = evaluate([test_case], [metric])

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4o, strict=False, async_mode=True)...

Output()

**************************************************

Contextual Precision Verbose Logs

**************************************************

Verdicts:
[
    {
        "verdict": "no",
        "reason": "The context 'Machine Learning is the study of algorithms which learn with more data' does not 
directly address what AI is or its applications."
    },
    {
        "verdict": "yes",
        "reason": "The context 'AI is known as Artificial Intelligence' directly defines AI, which is part of the 
expected output."
    },
    {
        "verdict": "yes",
        "reason": "The context 'Artificial intelligence refers to machines mimicking human intelligence, like 
problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles.'
provides a comprehensive explanation of AI and its applications, aligning with the expected output."
    },
    {
        "verdict": "yes",
        "reason": "The context 'Artificial intelligence refers to machines mimicking human intelligence, like 
problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles.'
is repeated and provides relevant information about AI and its applications."
    },
    {
        "verdict": "no",
        "reason": "The context 'NLP is a branch of AI that enables computers to understand, interpret, and generate
human language.' focuses on NLP, a specific branch of AI, rather than AI as a whole."
    }
]
 
Score: 0.6388888888888888
Reason: The score is 0.64 because the relevant nodes are generally ranked higher than the irrelevant ones, but 
there are still some improvements needed. The first node, ranked 1, is irrelevant as it discusses 'Machine 
Learning' without directly addressing AI, yet it precedes relevant nodes. The second node, ranked 2, correctly 
defines AI, aligning with the input. The third and fourth nodes, ranked 3 and 4, provide a comprehensive 
explanation of AI and its applications, which is highly relevant. However, the fifth node, ranked 5, focuses on 
'NLP', a specific branch of AI, rather than AI as a whole, and should be ranked lower. Overall, while the relevant 
nodes are present, the initial placement of irrelevant nodes affects the score.

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score        ┃ Pass Rate                                   ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Contextual Precision           │ 0.64                 │ 100.00% | passed=1 | failed=0               │ 1        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=9901180;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.57s | token cost: 0.007177500000000001 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [7]:
print(result)

test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Contextual Precision', threshold=0.5, success=True, score=0.6791666666666667, reason="The score is 0.68 because the first node, ranked highest, contains the context 'Machine Learning is the study of algorithms which learn with more data', which does not directly address what AI is or its applications. This irrelevant node should be ranked lower than the subsequent nodes, which provide direct definitions and detailed explanations of AI, such as the second node, which states 'AI is known as Artificial Intelligence', and the third node, which elaborates on AI's applications like virtual assistants and robotics. The presence of these relevant nodes in lower ranks affects the overall precision score.", strict_mode=False, evaluation_model='gpt-4o', error=None, evaluation_cost=0.006902500000000001, verbose_logs='Verdicts:\n[\n    {\n        "verdict": "no",\n        "reason": "The context \'Machine Learn

In [8]:
print('Sucess:', result.test_results[0].metrics_data[0].success)
print('Score:', result.test_results[0].metrics_data[0].score)
print('Reason:', result.test_results[0].metrics_data[0].reason)

Sucess: True
Score: 0.6791666666666667
Reason: The score is 0.68 because the first node, ranked highest, contains the context 'Machine Learning is the study of algorithms which learn with more data', which does not directly address what AI is or its applications. This irrelevant node should be ranked lower than the subsequent nodes, which provide direct definitions and detailed explanations of AI, such as the second node, which states 'AI is known as Artificial Intelligence', and the third node, which elaborates on AI's applications like virtual assistants and robotics. The presence of these relevant nodes in lower ranks affects the overall precision score.


## Contextual Recall

The contextual recall metric measures the quality of your RAG pipeline's retriever by evaluating the extent of which the `retrieval_context` aligns with the `expected_output`.

`deepeval`'s contextual recall metric is a self-explaining LLM-Eval, meaning it outputs a reason for its metric score using an LLM as a Judge.

In `deepeval`, to use the ContextualRecallMetric, you'll have to provide the following arguments when creating an `LLMTestCase`:

- `input` : Input Query (not used in the computation)
- `actual_output` : Actual LLM Response (not used in the computation)
- `expected_output` : Expected LLM Response (ground truth answer)
- `retrieval_context` : Top-N retrieved document chunks (nodes) from Vector DB


![](https://i.imgur.com/PDbwuX5.png)





In [7]:
query = "What is AI?"
response = rag_chain_w_sources.invoke(query)
response

{'context': [Document(id='ca8293b0-0909-4d6d-81b3-1b1568851c4e', metadata={'id': 10, 'title': 'Artificial Intelligence'}, page_content="Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning."),
  Document(id='453580ed-61d6-4f43-a237-7432eebc2be5', metadata={'id': 10, 'title': 'Artificial Intelligence'}, page_content="Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning."),
  Document(id='69522413-c9ae-41cf-b1b5-cc4e288dc58e', metadata={'id': 3, 'title': 'Natural Language Processing (NLP)'}, page_content='NLP is a branch of AI that enables computers to underst

### Example 1:

In [8]:
retrieved_context = [doc.page_content for doc in response['context']]
retrieved_context

["Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.']

In [9]:
retrieved_context

["Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.']

In [10]:
human_answer = """AI, also known as Artificial Intelligence is used to build complex systems for applications
                  like virtual assistants, robotics and autonomous vehicles."""

In [11]:
new_context = ['NVIDIA makes chips for AI', 'AI is an acronym for Artificial Intellence']
new_context

['NVIDIA makes chips for AI', 'AI is an acronym for Artificial Intellence']

In [12]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRecallMetric
from deepeval import evaluate

test_case1 = LLMTestCase(
    input=response['question'],
    actual_output=response['response'],
    expected_output=human_answer,
    retrieval_context=retrieved_context
)

test_case2 = LLMTestCase(
    input=response['question'],
    actual_output=response['response'],
    expected_output=human_answer,
    retrieval_context=new_context
)

metric = ContextualRecallMetric(
    threshold=0.5,
    model="gpt-4o",
    include_reason=True,
    verbose_mode=True
)

result = evaluate([test_case1, test_case2], [metric])

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4o, strict=False, async_mode=True)...

Output()

**************************************************

Contextual Recall Verbose Logs

**************************************************

...

Verdicts:
[
    {
        "verdict": "yes",
        "reason": "The sentence mentions 'AI, also known as Artificial Intelligence', which aligns with the 2nd 
node: 'AI is an acronym for Artificial Intelligence'.",
        "expected_output": "AI, also known as Artificial Intelligence is used to build complex systems for 
applications\n                  like virtual assistants, robotics and autonomous vehicles."
    },
    {
        "verdict": "no",
        "reason": "The sentence discusses applications like 'virtual assistants, robotics and autonomous vehicles',
which are not mentioned in the retrieval context.",
        "expected_output": "AI, also known as Artificial Intelligence is used to build complex systems for 
applications\n                  like virtual assistants, robotics and autonomous vehicles."
    }
]
 
Score: 0.5
Reason: The score is 0.50 because while the mention of 'AI, also known as Artificial Intelligence' aligns with node
2 in the retrieval context, the applications such as 'virtual assistants, robotics, and autonomous vehicles' are 
not supported by any nodes in the retrieval context.

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                      ┃ Average Score         ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Contextual Recall           │ 0.75                  │ 100.00% | passed=2 | failed=0                │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=9901182;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.26s | token cost: 0.007000000000000001 USD)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [15]:
print('Sucess:', result.test_results[0].metrics_data[0].success)
print('Score:', result.test_results[0].metrics_data[0].score)
print('Reason:', result.test_results[0].metrics_data[0].reason)

Sucess: True
Score: 1.0
Reason: The score is 1.00 because the expected output perfectly aligns with the information in the 1st node in the retrieval context, showcasing a seamless match. Great job!


In [16]:
print('Sucess:', result.test_results[1].metrics_data[0].success)
print('Score:', result.test_results[1].metrics_data[0].score)
print('Reason:', result.test_results[1].metrics_data[0].reason)

Sucess: True
Score: 0.5
Reason: The score is 0.50 because while the mention of 'AI, also known as Artificial Intelligence' aligns with the 2nd node in the retrieval context, the applications such as 'virtual assistants, robotics, and autonomous vehicles' are not supported by any node in the retrieval context.


## Contextual Relevancy

The contextual relevancy metric measures the quality of your RAG pipeline's retriever by evaluating the overall relevance of the information presented in your `retrieval_context` for a given `input`.

`deepeval`'s contextual relevancy metric is a self-explaining LLM-Eval, meaning it outputs a reason for its metric score using an LLM as a Judge.

In `deepeval`, to use the ContextualRelevancyMetric, you'll have to provide the following arguments when creating an `LLMTestCase`:

- `input` : Input Query
- `actual_output` : Actual LLM Response (not used in the computation)
- `retrieval_context` : Top-N retrieved document chunks (nodes) from Vector DB


![](https://i.imgur.com/VLKoEsI.png)





In [ ]:
query = "What is AI?"
response = rag_chain_w_sources.invoke(query)
response

### Example 1:

In [ ]:
retrieved_context = [doc.page_content for doc in response['context']]
retrieved_context

In [ ]:
new_context = ['NVIDIA makes chips for AI', 'Google and Microsoft are battling out the market share for AI Chatbots'] + retrieved_context
new_context

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRelevancyMetric
from deepeval import evaluate

test_case = LLMTestCase(
    input=response['question'],
    actual_output=response['response'],
    expected_output=human_answer,
    retrieval_context=new_context
)

metric = ContextualRelevancyMetric(
    threshold=0.5,
    model="gpt-4o",
    include_reason=True,
    verbose_mode=True
)

result = evaluate([test_case], [metric])

In [ ]:
print('Sucess:', result.test_results[0].metrics_data[0].success)
print('Score:', result.test_results[0].metrics_data[0].score)
print('Reason:', result.test_results[0].metrics_data[0].reason)